# Módulo 8 · Clase 7 — Redes Convolucionales (CNN): Visión Artificial

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Por qué las redes densas fallan con imágenes |
| 2 | La convolución: detectar patrones locales |
| 3 | Pooling: reducir dimensión |
| 4 | Arquitectura CNN completa en Keras |
| 5 | Transfer Learning: reutilizar redes preentrenadas |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from scipy.signal import convolve2d

tf.random.set_seed(42)
np.random.seed(42)
print(f"TensorFlow: {tf.__version__}")

---
## 1. Por qué las redes densas fallan con imágenes

```
Una imagen de 224×224×3 pixels (RGB) tiene 150,528 valores.

Red densa: 150,528 → 1,000 neuronas = 150,528,000 parámetros ¡solo en la primera capa!
Problemas:
  1. Demasiados parámetros → overfitting, memoria, lentitud
  2. Ignora la estructura espacial: no sabe que pixels cercanos están relacionados
  3. No es invariante a la traslación: un 'gato' en la esquina ≠ 'gato' en el centro

Las CNN aprovechan tres propiedades de las imágenes:
  1. Localidad: un ojo es un grupo de pixels cercanos
  2. Jerarquía: bordes → formas → objetos
  3. Traslación: el mismo detector de bordes sirve en todo el imagen

Solución: compartir pesos mediante filtros convolucionales
  Mismo filtro aplicado en cada posición → muchos menos parámetros
```

In [ ]:
# Comparar parámetros: Dense vs CNN para una imagen 28×28
print("Comparación de parámetros: imagen 28×28 (MNIST)")
print("="*55)

# Red densa
m_dense = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
print(f"Red Densa:")
m_dense.summary()

print("\nRed CNN:")
# CNN
m_cnn = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),          # imagen 2D + canal
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
m_cnn.summary()

---
## 2. La convolución: detectar patrones locales

```
Convolución 2D:
  Filtro (kernel): pequeña matriz, por ejemplo 3×3
  Se desliza por la imagen calculando el producto punto en cada posición
  Salida: feature map → resalta dónde se detectó el patrón del filtro

  Imagen 28×28 * Filtro 3×3 → Feature map 26×26 (sin padding)
  Con padding='same' → mantiene 28×28

Conv2D(32, (3,3)):
  32 filtros distintos, cada uno de 3×3
  Parámetros: 3×3×n_canales_entrada × 32 + 32 (bias)
  Para entrada RGB: 3×3×3×32 + 32 = 896 params
  Para MNIST (1 canal): 3×3×1×32 + 32 = 320 params

¿Qué detectan los filtros?
  Capa 1: bordes horizontales, verticales, diagonales
  Capa 2: curvas, esquinas, texturas
  Capa 3+: partes de objetos, objetos completos
```

In [ ]:
# Filtros clásicos para detectar bordes
filtros = {
    'Horizontal (Sobel)': np.array([[-1,-2,-1], [0,0,0], [1,2,1]]),
    'Vertical (Sobel)':   np.array([[-1,0,1],  [-2,0,2], [-1,0,1]]),
    'Diagonal':           np.array([[2,1,-1],  [1,0,-1], [-1,-1,2]]),
    'Nitidez':            np.array([[0,-1,0],  [-1,5,-1], [0,-1,0]]),
    'Blur':               np.ones((3,3)) / 9,
}

# Cargar una imagen de MNIST como ejemplo
(X_tr, _), _ = keras.datasets.mnist.load_data()
img = X_tr[0].astype('float32') / 255.0  # 28×28

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes[0,0].imshow(img, cmap='gray'); axes[0,0].set_title('Original'); axes[0,0].axis('off')

for ax, (nombre, f) in zip(axes.ravel()[1:], filtros.items()):
    feature_map = convolve2d(img, f, mode='same')
    ax.imshow(feature_map, cmap='RdBu_r')
    ax.set_title(f'Filtro: {nombre}', fontsize=9)
    ax.axis('off')

plt.suptitle('Convoluciones con filtros artesanales', fontsize=14)
plt.tight_layout(); plt.show()
print("→ Una CNN aprende estos filtros automáticamente desde los datos")

In [ ]:
# Conv2D en Keras: parámetros explicados
print("""
layers.Conv2D(
    filters=32,           # nº de filtros (feature maps de salida)
    kernel_size=(3,3),    # tamaño del filtro. Puede ser solo 3 (cuadrado)
    strides=(1,1),        # paso del deslizamiento (default)
    padding='valid',      # 'valid' = sin padding, 'same' = mantiene dimensión
    activation='relu',    # activación aplicada al feature map
    use_bias=True,        # sesgo por filtro
    input_shape=(H,W,C)   # en la primera capa: altura, ancho, canales
)

Salida con padding='valid':
  (H - k + 1) × (W - k + 1) × filters
  28×28 → Conv2D(32, 3) → 26×26×32

Salida con padding='same':
  H × W × filters
  28×28 → Conv2D(32, 3, padding='same') → 28×28×32

Strides:
  Conv2D(32, 3, strides=2) → salida ≈ H/2 × W/2 × 32 (reduce resolución)
""")

# Calcular parámetros de una capa Conv2D
for c_in, n_filtros, k in [(1, 32, 3), (32, 64, 3), (3, 128, 3)]:
    params = (k*k*c_in + 1) * n_filtros
    print(f"Conv2D({n_filtros}, {k}×{k}) con {c_in} canal(es) entrada: {params:,} parámetros")

---
## 3. Pooling: reducir dimensión

In [ ]:
print("""
MaxPooling2D(pool_size=(2,2), strides=None):
  Divide el feature map en bloques 2×2 y toma el máximo de cada bloque.
  Reduce resolución a la mitad: 26×26 → 13×13
  Ventaja: invarianza a pequeñas traslaciones
  Parámetros: 0 (no hay parámetros que aprender)

AveragePooling2D: promedio en lugar de máximo
GlobalAveragePooling2D: promedia TODA la dimensión espacial → 1 valor por canal
  Útil en la última capa convolucional antes de Dense
""")

# Visualizar MaxPooling
np.random.seed(0)
feat_map = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [1, 2, 8, 9],
    [3, 4, 6, 7]
], dtype=float)

pool = np.array([
    [max(feat_map[0,0], feat_map[0,1], feat_map[1,0], feat_map[1,1]),
     max(feat_map[0,2], feat_map[0,3], feat_map[1,2], feat_map[1,3])],
    [max(feat_map[2,0], feat_map[2,1], feat_map[3,0], feat_map[3,1]),
     max(feat_map[2,2], feat_map[2,3], feat_map[3,2], feat_map[3,3])],
])

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im1 = axes[0].imshow(feat_map, cmap='Blues', vmin=0)
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, int(feat_map[i,j]), ha='center', va='center', fontsize=16)
# marcar bloques
for i in range(2):
    for j in range(2):
        rect = plt.Rectangle((j*2-0.5, i*2-0.5), 2, 2,
                              linewidth=3, edgecolor='red', facecolor='none')
        axes[0].add_patch(rect)
axes[0].set_title('Feature map 4×4')

im2 = axes[1].imshow(pool, cmap='Blues', vmin=0)
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, int(pool[i,j]), ha='center', va='center',
                     fontsize=22, fontweight='bold')
axes[1].set_title('Después de MaxPooling(2×2) → 2×2')

plt.suptitle('Max Pooling: toma el máximo en cada bloque 2×2', fontsize=12)
plt.tight_layout(); plt.show()

---
## 4. Arquitectura CNN completa en Keras

In [ ]:
# CNN para MNIST
(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()

# Las CNN necesitan shape (H, W, C) — añadir dimensión de canal
X_tr_cnn = X_tr.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_te_cnn = X_te.reshape(-1, 28, 28, 1).astype('float32') / 255.0

print(f"Shape para CNN: {X_tr_cnn.shape}  (N, H, W, C)")

tf.random.set_seed(42)
modelo_cnn = keras.Sequential([
    # Bloque conv 1: detectar patrones básicos
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),  # 28×28×32
    layers.Conv2D(32, 3, activation='relu'),                   # 26×26×32
    layers.MaxPooling2D(2),                                    # 13×13×32
    layers.Dropout(0.25),

    # Bloque conv 2: detectar patrones compuestos
    layers.Conv2D(64, 3, activation='relu', padding='same'),  # 13×13×64
    layers.Conv2D(64, 3, activation='relu'),                   # 11×11×64
    layers.MaxPooling2D(2),                                    # 5×5×64
    layers.Dropout(0.25),

    # Clasificador
    layers.Flatten(),                                          # 5×5×64 = 1600
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax'),
], name='CNN_MNIST')

modelo_cnn.summary()

In [ ]:
modelo_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

h_cnn = modelo_cnn.fit(
    X_tr_cnn, y_tr,
    epochs=12, batch_size=128,
    validation_split=0.1,
    verbose=1
)

_, acc_cnn = modelo_cnn.evaluate(X_te_cnn, y_te, verbose=0)
print(f"\nCNN — Test accuracy: {acc_cnn:.4f} ({acc_cnn:.2%})")

# Comparar con Dense de la clase anterior (~97.5%)
# CNN habitual: >99%

In [ ]:
# Visualizar los filtros aprendidos por la primera capa
W_conv = modelo_cnn.layers[0].get_weights()[0]  # (3, 3, 1, 32)
print(f"Filtros de la capa Conv2D(32): {W_conv.shape}")

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.ravel()):
    filtro = W_conv[:, :, 0, i]  # filtro i
    ax.imshow(filtro, cmap='RdBu_r')
    ax.axis('off')
plt.suptitle('32 filtros aprendidos por la 1ª capa Conv2D\n(azul=activa, rojo=inhibe)', fontsize=12)
plt.tight_layout(); plt.show()

# Visualizar feature maps para una imagen
modelo_conv1 = keras.Model(inputs=modelo_cnn.input, outputs=modelo_cnn.layers[0].output)
img = X_te_cnn[0:1]  # un dígito
feature_maps = modelo_conv1.predict(img, verbose=0)  # (1, 28, 28, 32)

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(feature_maps[0, :, :, i], cmap='viridis')
    ax.axis('off')
plt.suptitle(f'Feature maps (salida capa Conv1) para el dígito {y_te[0]}', fontsize=12)
plt.tight_layout(); plt.show()

---
## 5. Transfer Learning: reutilizar redes preentrenadas

```
Problema: entrenar una CNN grande desde cero requiere:
  Millones de imágenes + semanas de cómputo en GPU

Transfer Learning: usar una red ya entrenada en ImageNet (1.2M imágenes, 1000 clases)
  Las capas convolucionales aprenden detectores de bordes, texturas, formas...
  Estos detectores son ÚTILES para cualquier tarea de visión

Estrategias:
  1. Feature extraction: congela las capas conv, solo entrena el clasificador final
  2. Fine-tuning: descongela las últimas capas conv y entrena con lr muy baja

Modelos disponibles en Keras:
  keras.applications.VGG16
  keras.applications.ResNet50
  keras.applications.MobileNetV2    ← ligero, ideal para producción
  keras.applications.EfficientNetB0 ← equilibrio rendimiento/velocidad
  keras.applications.InceptionV3
```

In [ ]:
# Transfer Learning con MobileNetV2 para clasificación binaria
# Usamos CIFAR-10: perros (clase 5) vs gatos (clase 3)
(X_tr_c, y_tr_c), (X_te_c, y_te_c) = keras.datasets.cifar10.load_data()

# Filtrar solo perros y gatos
mask_tr = (y_tr_c.ravel() == 3) | (y_tr_c.ravel() == 5)
mask_te = (y_te_c.ravel() == 3) | (y_te_c.ravel() == 5)

X_tr_gc = X_tr_c[mask_tr].astype('float32') / 255.0
y_tr_gc = (y_tr_c[mask_tr].ravel() == 5).astype('float32')  # 1=perro, 0=gato
X_te_gc = X_te_c[mask_te].astype('float32') / 255.0
y_te_gc = (y_te_c[mask_te].ravel() == 5).astype('float32')

print(f"Perros vs Gatos — Train: {X_tr_gc.shape}, Test: {X_te_gc.shape}")
print(f"Clases: {np.bincount(y_tr_gc.astype(int))} (gato, perro)")

# CIFAR-10 es 32×32, MobileNetV2 necesita mínimo 96×96
# Usamos GlobalAveragePooling y una CNN simple para no necesitar resize

# Mostrar algunas imágenes
clases_gc = ['Gato', 'Perro']
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, (cls, label) in enumerate(zip([0, 1], ['Gatos', 'Perros'])):
    idx = np.where(y_tr_gc == cls)[0][:8]
    for ax, j in zip(axes[i], idx):
        ax.imshow(X_tr_gc[j])
        ax.axis('off')
    axes[i, 0].set_ylabel(label, rotation=90, va='center', fontsize=12)
plt.suptitle('CIFAR-10: Gatos vs Perros', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# CNN desde cero para gatos vs perros (baseline)
tf.random.set_seed(42)
modelo_cifar = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),

    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid'),
], name='CNN_cifar_scratch')

modelo_cifar.compile('adam', 'binary_crossentropy', ['accuracy'])

h_cifar = modelo_cifar.fit(
    X_tr_gc, y_tr_gc, epochs=25, batch_size=64,
    validation_split=0.15,
    callbacks=[keras.callbacks.EarlyStopping('val_loss', patience=8, restore_best_weights=True)],
    verbose=1
)

_, acc_cifar = modelo_cifar.evaluate(X_te_gc, y_te_gc, verbose=0)
print(f"\nCNN desde cero — Test accuracy: {acc_cifar:.4f}")

In [ ]:
# Transfer Learning: MobileNetV2 preentrenado en ImageNet
# Necesitamos redimensionar a 96×96 mínimo
X_tr_up = tf.image.resize(X_tr_gc, (96, 96)).numpy()
X_te_up = tf.image.resize(X_te_gc, (96, 96)).numpy()

# Cargar base preentrenada (sin el clasificador final)
base_model = keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,      # sin la capa Dense final
    weights='imagenet'      # pesos preentrenados
)
base_model.trainable = False  # congelar todos los pesos

print(f"MobileNetV2 base: {base_model.count_params():,} parámetros (congelados)")

# Añadir nuestro clasificador encima
entradas  = keras.Input(shape=(96, 96, 3))
x = keras.applications.mobilenet_v2.preprocess_input(entradas * 255)  # normalizar para MV2
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
salidas = layers.Dense(1, activation='sigmoid')(x)

modelo_tl = keras.Model(entradas, salidas, name='MobileNetV2_TL')
print(f"Parámetros entrenables: {sum(v.numpy().size for v in modelo_tl.trainable_variables):,}")

modelo_tl.compile('adam', 'binary_crossentropy', ['accuracy'])

h_tl = modelo_tl.fit(
    X_tr_up, y_tr_gc, epochs=10, batch_size=64,
    validation_split=0.15, verbose=1
)

_, acc_tl = modelo_tl.evaluate(X_te_up, y_te_gc, verbose=0)
print(f"\nTransfer Learning (MobileNetV2) — Test accuracy: {acc_tl:.4f}")
print(f"CNN desde cero                   — Test accuracy: {acc_cifar:.4f}")
print(f"Mejora con Transfer Learning: +{acc_tl-acc_cifar:.4f}")

---
## Práctica

### Ejercicio 1 — CNN para Fashion MNIST

**Enunciado:** Fashion MNIST es más exigente que MNIST — ideal para comparar Dense vs CNN.

1. Carga y preprocesa (28×28×1)
2. Arquitectura: 2 bloques conv + clasificador Dense
3. Añade Dropout y BatchNorm
4. Compara con el modelo Dense de la clase anterior
5. ¿Cuántos parámetros tiene cada uno?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr_cnn = X_tr.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_te_cnn = X_te.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_tr_d   = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te_d   = X_te.reshape(-1, 784).astype('float32') / 255.0

# CNN
tf.random.set_seed(42)
m_cnn = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax'),
])
m_cnn.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
m_cnn.fit(X_tr_cnn, y_tr, epochs=15, batch_size=128, validation_split=0.1, verbose=0)
_, acc_cnn = m_cnn.evaluate(X_te_cnn, y_te, verbose=0)

# Dense
tf.random.set_seed(42)
m_dense = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10,  activation='softmax'),
])
m_dense.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
m_dense.fit(X_tr_d, y_tr, epochs=15, batch_size=128, validation_split=0.1, verbose=0)
_, acc_dense = m_dense.evaluate(X_te_d, y_te, verbose=0)

print(f"CNN   — params: {m_cnn.count_params():,}, acc: {acc_cnn:.4f}")
print(f"Dense — params: {m_dense.count_params():,}, acc: {acc_dense:.4f}")
print(f"CNN usa {m_cnn.count_params()/m_dense.count_params():.1f}× menos parámetros y logra mejor resultado")
```
</details>

---
### Ejercicio 2 — Visualizar activaciones

**Enunciado:** ¿Qué ve la red en cada capa?

1. Toma el modelo CNN de MNIST ya entrenado
2. Usa `keras.Model(inputs=..., outputs=capa.output)` para extraer activaciones de cada bloque
3. Muestra los feature maps de las capas: Conv1, Conv2, Conv3 y Conv4
4. ¿Cómo cambia la 'representación' a lo largo de las capas?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import matplotlib.pyplot as plt
import numpy as np

# Capa conv layers del modelo ya entrenado
capas_conv = [l for l in modelo_cnn.layers if 'conv' in l.name]
print(f"Capas Conv: {[l.name for l in capas_conv]}")

# Imagen de ejemplo
img = X_te_cnn[0:1]  # (1, 28, 28, 1)
print(f"Clase real: {y_te[0]}")

fig = plt.figure(figsize=(16, 10))
for row, capa in enumerate(capas_conv):
    m_interm = tf.keras.Model(inputs=modelo_cnn.input, outputs=capa.output)
    feat = m_interm.predict(img, verbose=0)  # (1, H, W, n_filtros)
    n_show = min(8, feat.shape[-1])

    for col in range(n_show):
        ax = fig.add_subplot(len(capas_conv), n_show, row*n_show + col + 1)
        ax.imshow(feat[0, :, :, col], cmap='viridis')
        if col == 0:
            ax.set_ylabel(capa.name[:10], fontsize=8, rotation=0, labelpad=40, va='center')
        ax.axis('off')

plt.suptitle(f'Feature maps por capa para dígito {y_te[0]}', fontsize=13)
plt.tight_layout(); plt.show()
print("→ Capa 1: bordes crudos. Capa 2: formas. Capas profundas: patrones abstractos.")
```
</details>

---
### Ejercicio 3 — Data Augmentation

**Enunciado:** Ampliar el dataset artificialmente con transformaciones.

```python
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])
```

1. Solo 2000 muestras de Fashion MNIST
2. Añade data augmentation como primer bloque del modelo
3. Compara con el mismo modelo sin augmentation
4. ¿Mejora la generalización?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
N = 2000
X_tr_n = X_tr[:N].reshape(-1, 28, 28, 1).astype('float32') / 255.0
y_tr_n = y_tr[:N]
X_te_n = X_te.reshape(-1, 28, 28, 1).astype('float32') / 255.0

def build_cnn(augmentation=False):
    capas = [layers.Input(shape=(28, 28, 1))]
    if augmentation:
        capas += [
            layers.RandomFlip('horizontal'),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1),
        ]
    capas += [
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(10, activation='softmax'),
    ]
    m = keras.Sequential(capas)
    m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
    return m

for nombre, aug in [('Sin augmentation', False), ('Con augmentation', True)]:
    tf.random.set_seed(42)
    m = build_cnn(aug)
    h = m.fit(X_tr_n, y_tr_n, epochs=30, batch_size=64,
              validation_split=0.15, verbose=0)
    _, acc_te = m.evaluate(X_te_n, y_te, verbose=0)
    acc_tr  = h.history['accuracy'][-1]
    acc_val = h.history['val_accuracy'][-1]
    print(f"{nombre}: tr={acc_tr:.4f}, val={acc_val:.4f}, te={acc_te:.4f}, gap={acc_tr-acc_val:.4f}")
```
</details>

---
### Ejercicio 4 — Fine-tuning

**Enunciado:** El Transfer Learning con base congelada no usa todo el potencial del modelo preentrenado. El Fine-tuning descongela las últimas capas.

1. Usa el modelo `modelo_tl` (MobileNetV2) ya entrenado
2. Descongela las últimas 20 capas de `base_model`
3. Reentrena con lr muy baja (1e-5) durante 5 épocas
4. ¿Mejora la accuracy en gatos vs perros?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
# Accuracy antes del fine-tuning (ya entrenado arriba)
acc_antes = modelo_tl.evaluate(X_te_up, y_te_gc, verbose=0)[1]
print(f"Antes del fine-tuning: {acc_antes:.4f}")

# Descongelar las últimas 20 capas
base_model.trainable = True
for capa in base_model.layers[:-20]:
    capa.trainable = False

n_train = sum(v.numpy().size for v in modelo_tl.trainable_variables)
print(f"Parámetros entrenables tras descongelar últimas 20 capas: {n_train:,}")

# Recompilar con lr muy baja (imprescindible en fine-tuning)
modelo_tl.compile(
    optimizer=keras.optimizers.Adam(1e-5),  # lr muy baja: no destruir los pesos
    loss='binary_crossentropy',
    metrics=['accuracy']
)

h_ft = modelo_tl.fit(
    X_tr_up, y_tr_gc, epochs=5, batch_size=32,
    validation_split=0.15, verbose=1
)

acc_despues = modelo_tl.evaluate(X_te_up, y_te_gc, verbose=0)[1]
print(f"\nAnte del fine-tuning:   {acc_antes:.4f}")
print(f"Después del fine-tuning: {acc_despues:.4f}")
print(f"Mejora: +{acc_despues-acc_antes:.4f}")
```
</details>

---
### Ejercicio 5 (reto) — GradCAM: ¿dónde mira la red?

**Enunciado:** GradCAM (Gradient-weighted Class Activation Mapping) muestra qué región de la imagen influye más en la predicción.

1. Implementa una función `gradcam(model, img, clase_idx, nombre_capa)` que:
   - Calcule los gradientes de la clase respecto a los feature maps de la última capa conv
   - Genere el mapa de calor
   - Superponga el mapa sobre la imagen original
2. Aplícalo a 5 imágenes de MNIST
3. ¿En qué parte del dígito se fija la red para predecir cada clase?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

def gradcam(model, img, clase_idx, nombre_capa):
    """img: (1, H, W, C) normalizada"""
    capa = model.get_layer(nombre_capa)
    grad_model = tf.keras.Model(
        inputs=model.input,
        outputs=[capa.output, model.output]
    )
    with tf.GradientTape() as tape:
        feat_maps, preds = grad_model(img)
        score = preds[0, clase_idx]

    grads   = tape.gradient(score, feat_maps)      # (1, H, W, n_filtros)
    pooled  = tf.reduce_mean(grads, axis=(0,1,2))  # (n_filtros,)
    heatmap = tf.reduce_mean(feat_maps[0] * pooled, axis=-1)  # (H, W)
    heatmap = tf.nn.relu(heatmap).numpy()
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
    return heatmap

# Última capa conv del modelo MNIST
capas_conv = [l.name for l in modelo_cnn.layers if 'conv' in l.name]
ultima_conv = capas_conv[-1]
print(f"Usando capa: {ultima_conv}")

# 5 imágenes de distintas clases
n_clases = 5
fig, axes = plt.subplots(n_clases, 3, figsize=(10, 3*n_clases))

for i, clase in enumerate(range(n_clases)):
    idx = np.where(y_te == clase)[0][0]
    img = X_te_cnn[idx:idx+1]

    heatmap = gradcam(modelo_cnn, img, clase, ultima_conv)
    heatmap_resized = tf.image.resize(heatmap[..., None], (28, 28)).numpy()[:,:,0]

    # Original
    axes[i,0].imshow(img[0,:,:,0], cmap='gray')
    axes[i,0].set_title(f'Dígito real: {clase}'); axes[i,0].axis('off')

    # Heatmap
    axes[i,1].imshow(heatmap_resized, cmap='hot')
    axes[i,1].set_title('GradCAM'); axes[i,1].axis('off')

    # Superposición
    img_rgb = np.stack([img[0,:,:,0]]*3, axis=-1)
    hm_rgb  = plt.cm.jet(heatmap_resized)[:,:,:3]
    overlay = 0.6 * img_rgb + 0.4 * hm_rgb
    overlay = np.clip(overlay, 0, 1)
    axes[i,2].imshow(overlay)
    axes[i,2].set_title('Superposición'); axes[i,2].axis('off')

plt.suptitle('GradCAM: donde mira la red para predecir cada dígito', fontsize=13)
plt.tight_layout(); plt.show()
```
</details>

---
## Resumen

```python
# CNN típica en Keras
modelo = keras.Sequential([
    layers.Input(shape=(H, W, C)),   # C=1 gris, C=3 RGB

    # Bloque convolucional
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(2),           # ÷2 en espacial
    layers.Dropout(0.25),

    # Más bloques...

    # Clasificador
    layers.Flatten(),                 # o GlobalAveragePooling2D()
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(n_clases, activation='softmax'),
])

# Transfer Learning
base = keras.applications.MobileNetV2(include_top=False, weights='imagenet')
base.trainable = False
# ... añadir clasificador encima
```

| Capa | Función | Parámetros |
|------|---------|------------|
| Conv2D(f, k) | Detectar patrones locales | k²·C·f + f |
| MaxPooling2D(2) | Reducir resolución ÷2 | 0 |
| GlobalAveragePooling2D | Resumir feature map a 1 valor | 0 |
| Flatten | Aplanar a 1D | 0 |

## ¿Qué viene en la próxima clase?
En la **Clase 8** aprenderemos **RNN y Transformers** — arquitecturas para secuencias (texto, audio, series temporales) con LSTM, GRU y el mecanismo de atención.